# Notebook 11 — Skills for the Research Deep Agent

Skills are reusable procedural knowledge for the `deep-agents-on-foundry` research agent.

## Mental model
- Memory = **what do I know?**
- Skill = **how do I perform this kind of task?**
- Tool = **what action can I execute?**
- Deep Agent = **which knowledge, procedure, and tools should I combine?**

## Sections
11.1 Skills foundations  
11.2 Skills vs prompts/tools/memory  
11.3 First research skill  
11.4 Progressive disclosure  
11.5 Skill routing  
11.6 Supporting files + layering  
11.7 Skills + memory  
11.8 Evaluation + economics

## 11.1 — Why Skills?

A general-purpose research agent may need procedures for technology research, company research,
architecture comparison, paper analysis, and evidence synthesis.

Putting all procedures in the system prompt bloats every model call.

```text
small core prompt
    + skill metadata catalog
          ↓
       user task
          ↓
  relevant skill selected
          ↓
   full SKILL.md loaded
```

This is progressive disclosure.

## 11.2 — Skills vs other mechanisms

| Mechanism | Main question |
|---|---|
| System prompt | How should I generally behave? |
| Memory | What do I know about the user/project? |
| Skill | How should I perform this task? |
| Tool | What action can I execute? |
| Checkpointer | What happened in this thread? |

## 11.3 — Inspect installed APIs first

In [1]:
import inspect
from deepagents import create_deep_agent
print(inspect.signature(create_deep_agent))

try:
    from deepagents.backends.filesystem import FilesystemBackend
    print("FilesystemBackend:", inspect.signature(FilesystemBackend))
except Exception as exc:
    print("FilesystemBackend unavailable or different:", exc)

(model: str | langchain_core.language_models.chat_models.BaseChatModel | None = None, tools: collections.abc.Sequence[langchain_core.tools.base.BaseTool | collections.abc.Callable | dict[str, typing.Any]] | None = None, *, system_prompt: str | langchain_core.messages.system.SystemMessage | None = None, middleware: collections.abc.Sequence[langchain.agents.middleware.types.AgentMiddleware[+StateT_co, ~ContextT, typing.Any]] = (), subagents: collections.abc.Sequence[deepagents.middleware.subagents.SubAgent | deepagents.middleware.subagents.CompiledSubAgent | deepagents.middleware.async_subagents.AsyncSubAgent] | None = None, skills: list[str] | None = None, memory: list[str] | None = None, permissions: list[deepagents.middleware.filesystem.FilesystemPermission] | None = None, backend: deepagents.backends.protocol.BackendProtocol | None = None, interrupt_on: dict[str, bool | langchain.agents.middleware.human_in_the_loop.InterruptOnConfig] | None = None, response_format: Union[langchain.ag

## 11.4 — Inspect the skill library

This bundle includes:
- technology-research
- company-research
- architecture-comparison
- research-paper-analysis
- evidence-synthesis

Start with only one or two active skills.

In [2]:
from pathlib import Path
skills_root = Path("../skills")

for path in sorted(skills_root.glob("*/SKILL.md")):
    print("="*80)
    print(path)
    print(path.read_text(encoding="utf-8"))

..\skills\architecture-comparison\SKILL.md
---
name: architecture-comparison
description: Compare system or agent architectures using quality, reliability, state ownership, complexity, latency, cost, portability, and operability.
---
# Architecture Comparison
1. Clarify workload and success criteria.
2. Draw each architecture simply.
3. Identify model, harness, runtime, persistence, tools, and platform boundaries.
4. Compare reliability and failure isolation.
5. Compare state/memory ownership.
6. Compare latency and cost.
7. Compare implementation/operational complexity.
8. Compare portability and lock-in.
9. Identify strengths of each.
10. Recommend based on workload.

..\skills\company-research\SKILL.md
---
name: company-research
description: Research a public company through business quality, competitive advantage, economics, cash generation, dilution, risks, and valuation-relevant evidence.
---
# Company Research
1. What does it sell and who pays?
2. Business model and revenue driv

## 11.5 — Skill anatomy

```text
technology-research/
├── SKILL.md
├── architecture_checklist.md
└── source_quality.md
```

`SKILL.md` is the compact procedural entry point.
Supporting files are deeper references loaded only when needed.

In [3]:
tech = skills_root / "technology-research"
print((tech/"SKILL.md").read_text())

---
name: technology-research
description: Research technologies and AI platforms using authoritative sources, architecture analysis, trade-offs, operations, and economics.
---
# Technology Research
1. Define scope.
2. Prefer primary sources.
3. Identify architecture components and data/control flow.
4. Identify state, identity, security, and trust boundaries.
5. Explain deployment and operations.
6. Surface reliability and failure modes.
7. Compare meaningful alternatives.
8. Explain trade-offs.
9. Discuss economics where relevant.
10. Synthesize rather than dump documentation.

Output: intuition first, then architecture, trade-offs, and implementation detail.



## 11.6 — Reuse the real research-agent components

In [ ]:
from deep_agents_foundry.agent import RESEARCH_INSTRUCTIONS
from deep_agents_foundry.model import build_model
from deep_agents_foundry.tools import build_web_search_tool
from deep_agents_foundry import build_sqlite_checkpointer, thread_config, content_text

model = build_model()
web_search = build_web_search_tool()
checkpointer = build_sqlite_checkpointer("../data/checkpoints.db")

## 11.7 — Build a Skills-enabled Deep Agent

Backend APIs can vary by installed version, so adapt this one cell after inspecting the signature.

Conceptually:

```python
backend = FilesystemBackend(root_dir=".")

skills_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=RESEARCH_INSTRUCTIONS,
    skills=["/skills/technology-research/"],
    backend=backend,
    checkpointer=checkpointer,
)
```

In [ ]:
# Uncomment/adapt after verifying your installed backend signature.
#
# backend = FilesystemBackend(root_dir=".")
# skills_agent = create_deep_agent(
#     model=model,
#     tools=[web_search],
#     system_prompt=RESEARCH_INSTRUCTIONS,
#     skills=["/skills/technology-research/"],
#     backend=backend,
#     checkpointer=checkpointer,
# )

## 11.8 — Progressive disclosure experiment

Ask:

> Research Microsoft Foundry Hosted Agents and explain architecture, identity,
> deployment model, trade-offs, and operational implications.

Observe whether the technology skill is discovered/read and then shapes the research.

In [ ]:
# result = skills_agent.invoke(
#     {"messages":[{"role":"user","content":
#       "Research Microsoft Foundry Hosted Agents and explain architecture, identity, "
#       "deployment model, trade-offs, and operational implications."}]},
#     config=thread_config("skill-tech-1"),
# )
# print(content_text(result["messages"][-1]))

## 11.9 — Skill routing

Add a second skill: `company-research`.

Expected routing examples:

```text
Research LangGraph architecture
→ technology-research

Research Adobe as a business
→ company-research

Say hello
→ ideally no research skill
```

Skill routing is a retrieval problem:
- memory retrieval = retrieve relevant knowledge/preferences
- skill retrieval = retrieve relevant procedure

In [ ]:
print((skills_root/"company-research"/"SKILL.md").read_text())

## 11.10 — Supporting files

The technology skill includes deeper references.

In [ ]:
for name in ["architecture_checklist.md", "source_quality.md"]:
    p = skills_root/"technology-research"/name
    print("="*80)
    print(name)
    print(p.read_text())

## 11.11 — Skill layering and overrides

Conceptually:

```text
base skills
   ↓
team skills
   ↓
project skills
   ↓
user/project overrides
```

This resembles configuration layering. Use overrides deliberately.

## 11.12 — What should become a Skill?

Good heuristic:

> If you could teach a new teammate a reusable multi-step playbook, it is probably a good Skill.

Good:
- technology-research
- company-research
- architecture-comparison
- paper-analysis
- evidence-synthesis

Poor:
- say hello
- generic one-step questions
- generic web search itself

## 11.13 — Skill vs Tool

```text
Tool
= primitive action

Skill
= reusable strategy/playbook
```

Example:
- Tool: `web_search(query)`
- Skill: search authoritative docs → inspect architecture → compare evidence → surface trade-offs → synthesize

## 11.14 — Skills + Memory

Example:

```text
Memory:
User prefers primary sources.

Skill:
Technology research procedure.

Tool:
Foundry Web Search.

Deep Agent:
combines preference + procedure + action.
```

This is more modular than a giant system prompt.

## 11.15 — Failure modes

Skills introduce:
- wrong skill selected
- relevant skill not selected
- too many skills loaded
- ambiguous descriptions
- conflicting instructions
- stale procedures
- over-reading supporting files
- token cost without quality gain
- over-agenting simple tasks

## 11.16 — Evaluate Skills

Evaluate **routing quality** and **outcome quality** separately.

Routing:
- should skill activate?
- did it activate?

Outcome:
- quality
- citations/source quality
- completeness
- tokens
- latency
- search count
- task success

### Small skill-routing dataset

| Prompt | Expected |
|---|---|
| Research LangGraph architecture | technology-research |
| Research Adobe as a business | company-research |
| Compare Foundry and AKS | architecture-comparison |
| Explain this paper | research-paper-analysis |
| Reconcile conflicting sources | evidence-synthesis |
| Say hello | none |

## 11.17 — Context economics

Naive approach:
```text
5 procedures × ~2,000 tokens
≈ 10,000 tokens always present
```

Skills approach:
```text
small metadata catalog
      ↓
relevant skill selected
      ↓
only needed procedure loaded
```

Skills trade **routing risk** for **context efficiency**.

## 11.18 — Recommended progression

Do not activate all five skills immediately.

```text
Experiment 1
technology-research only

Experiment 2
technology-research + company-research

Experiment 3
add architecture-comparison if routing is clean

Later
paper-analysis / evidence-synthesis
```

# Notebook 11 — Key takeaways

1. Skills encode reusable procedural knowledge.
2. Memory = what I know; Skill = how I work; Tool = what I can do.
3. Progressive disclosure reduces prompt bloat.
4. Skill metadata quality drives routing quality.
5. Supporting files enable deeper progressive disclosure.
6. Large catalogs can create ambiguity.
7. Evaluate both selection and outcome quality.
8. Skills are procedural retrieval + context engineering.
9. Skills must earn their token/latency cost.
10. This leads naturally into Middleware + Context Management.

Final mental model:

```text
Memory → relevant knowledge/preferences
Skills → relevant procedures
Tools  → actions
Deep Agent → coordinates all three
```